# Model Components Testing

Interactive checks for encoder, attention, SSL, heads, and losses.

Config composition follows the same style as `data_loading_testing.ipynb`.

In [10]:
import copy
import warnings

import anndata as ad
import hydra
import rootutils
import torch
from omegaconf import OmegaConf
from torch_geometric.data import Data

rootutils.setup_root("..", indicator=".project-root", pythonpath=True)

from src.data.spatial_omics_datamodule import SpatialOmicsDataModule
from src.models.components.factory import (
    build_attention,
    build_encoder,
    build_graph_head,
    build_loss,
    build_node_head,
    build_ssl,
)

with hydra.initialize_config_dir(
    config_dir="/Users/lovrorabuzin/Projects/grass-mil_unification/grass-mil/configs",
    version_base="1.3",
):
    cfg = hydra.compose(
        config_name="train",
        overrides=[
            "experiment=notebook_model_components",
            "hydra/job_logging=stdout",
            "hydra/hydra_logging=none",
        ],
    )

print(OmegaConf.to_yaml(cfg.model))

flags:
  use_attention: true
  use_ssl: true
encoder:
  _target_: src.models.components.backbones.EncoderConfig
  input_dim: 64
  hidden_dim: 128
  out_dim: 128
  num_layers: 3
  dropout: 0.1
  conv_type: gine
  norm: batchnorm
  jk: last
  act: relu
  pooling: mean
  set2set_steps: 3
  gat_heads: 4
  use_edge_attr: true
  edge_weight_index: 0
  edge_attr_dim: 2
attention:
  _target_: src.models.components.attention.AttnNetGatedProjected
  input_dim: 128
  projection_dim: 64
  hidden_dim: 16
  dropout: false
  n_classes: 1
ssl:
  method: bgrl
  predictor:
    _target_: src.models.components.ssl.MLPPredictor
    input_size: 128
    output_size: 128
    hidden_size: 256
graph_head:
  _target_: src.models.components.heads.GraphPredictionHead
  input_dim: 128
  output_dim: 1
  hidden_dim: 128
  num_layers: 2
  dropout: 0.0
node_head:
  _target_: src.models.components.heads.NodePredictionHead
  input_dim: 128
  output_dim: 1
  hidden_dim: 128
  num_layers: 2
  dropout: 0.0



## Notes

- Keep experiment-specific settings in `configs/experiment/notebook_model_components.yaml`.
- Change behavior by editing that experiment file, then re-run this notebook.

In [2]:
dm: SpatialOmicsDataModule = hydra.utils.instantiate(cfg.data)
dm.prepare_data()
dm.setup()

unit = dm.train_dataloader().dataset[0]
print(unit)
print(f"unit nodes={unit.num_nodes}, unit edges={unit.num_edges}")

Data(
  x=[762, 34],
  edge_index=[2, 4536],
  pos=[762, 2],
  edge_attr=[4536, 2],
  edge_attr_names=[2],
  categorical_index=[762, 1],
  categorical_labels=[1],
  categorical_slices={ cell type=0 },
  sample_id='s1',
  region_id='r1',
  patch_id='s1_r1_patch_0'
)
unit nodes=762, unit edges=4536


/Users/lovrorabuzin/Projects/grass-mil_unification/grass-mil/src/data/components/datasets.py:45: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(entry.path)


In [3]:
encoder_cfg = hydra.utils.instantiate(cfg.model.encoder)
encoder_cfg.input_dim = int(unit.x.size(1))

edge_attr = unit.edge_attr if hasattr(unit, "edge_attr") else None
if (
    encoder_cfg.use_edge_attr
    and edge_attr is not None
    and encoder_cfg.conv_type == "gine"
):
    encoder_cfg.edge_attr_dim = int(edge_attr.size(1))

encoder = build_encoder(encoder_cfg)
batch = getattr(unit, "batch", None)

node_emb, graph_emb = encoder(
    unit.x,
    unit.edge_index,
    edge_attr=edge_attr,
    batch=batch,
    return_graph_embedding=True,
)

print(f"input_dim used={encoder_cfg.input_dim}")
print(f"node_emb shape={tuple(node_emb.shape)}")
print(f"graph_emb shape={tuple(graph_emb.shape)}")

input_dim used=34
node_emb shape=(762, 128)
graph_emb shape=(1, 128)


In [4]:
graph_head_cfg = OmegaConf.to_container(cfg.model.graph_head, resolve=True)
graph_head_cfg.pop("_target_", None)
node_head_cfg = OmegaConf.to_container(cfg.model.node_head, resolve=True)
node_head_cfg.pop("_target_", None)

graph_head = build_graph_head(graph_head_cfg)
node_head = build_node_head(node_head_cfg)

print("graph logits shape:", tuple(graph_head(graph_emb).shape))
print("node logits shape:", tuple(node_head(node_emb).shape))

attention_cfg = OmegaConf.to_container(cfg.model.attention, resolve=True)
attention_cfg.pop("_target_", None)
attention = build_attention(
    use_attention=cfg.model.flags.use_attention,
    attention_type="gated_projected",
    **attention_cfg,
)
if attention is not None:
    attn_logits, _ = attention(node_emb)
    print("attention logits shape:", tuple(attn_logits.shape))
else:
    print("attention disabled")

graph logits shape: (1, 1)
node logits shape: (762, 1)
attention logits shape: (762, 1)


In [5]:
ssl_cfg = OmegaConf.to_container(cfg.model.ssl, resolve=True)
if "predictor" in ssl_cfg and isinstance(ssl_cfg["predictor"], dict):
    ssl_cfg["predictor"].pop("_target_", None)

ssl_module = build_ssl(
    use_ssl=cfg.model.flags.use_ssl,
    encoder=encoder,
    ssl_config=ssl_cfg,
)

if ssl_module is None:
    print("SSL disabled")
else:
    online_q, target_y = ssl_module(
        (unit.x, unit.edge_index, edge_attr),
        (unit.x, unit.edge_index, edge_attr),
    )
    print("online_q shape:", tuple(online_q.shape))
    print("target_y shape:", tuple(target_y.shape))

online_q shape: (762, 128)
target_y shape: (762, 128)


In [6]:
ce_loss = build_loss("categorical_ce")
ce_logits = torch.randn(8, 3)
ce_target = torch.randint(0, 3, (8,))
print("categorical_ce:", float(ce_loss(ce_logits, ce_target)))

mse_loss = build_loss("regression_mse")
y_pred = torch.randn(8, 1)
y_true = torch.randn(8, 1)
print("regression_mse:", float(mse_loss(y_pred, y_true)))

huber_loss = build_loss("regression_huber", delta=1.0)
print("regression_huber:", float(huber_loss(y_pred, y_true)))

cox_loss = build_loss("survival_coxsgd", top_n=2, regularizer_weight=0.05)
length = torch.tensor([5.0, 7.0, 2.0, 9.0, 6.0, 3.0, 8.0, 4.0])
event = torch.tensor([1.0, 1.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0])
print("survival_coxsgd:", float(cox_loss(torch.randn(8, 1), length, event)))

categorical_ce: 1.1970868110656738
regression_mse: 1.8591121435165405
regression_huber: 0.6854403018951416
survival_coxsgd: 2.9285335540771484


In [ ]:
## Optional quick synthetic check

# If you want to isolate from dataset-specific fields, you can run this in a code cell and then rerun cells 4-7:

x = torch.randn(64, encoder_cfg.input_dim)
edge_index = torch.randint(0, 64, (2, 256))
edge_attr_syn = torch.randn(256, encoder_cfg.edge_attr_dim or 1)
unit = Data(x=x, edge_index=edge_index, edge_attr=edge_attr_syn)

## Load the same H5AD as data notebook

This reads the exact file used in `data_loading_testing.ipynb` and confirms basic structure before graph modeling.

In [11]:
h5ad_path = "/Users/lovrorabuzin/Projects/grass-mil_unification/grass-mil/data/dummy/imc_data.h5ad"
adata = ad.read_h5ad(h5ad_path)
print(adata)
print(f"X shape: {adata.X.shape}")
print(f"obs columns: {list(adata.obs.columns)[:8]}")
if "spatial" in adata.obsm:
    print(f"obsm['spatial'] shape: {adata.obsm['spatial'].shape}")

AnnData object with n_obs × n_vars = 4668 × 34
    obs: 'cell type'
    uns: 'cell type_colors'
    obsm: 'spatial'
X shape: (4668, 34)
obs columns: ['cell type']
obsm['spatial'] shape: (4668, 2)


## Exhaustive model application matrix

Runs the model stack across:
- backbones: `gin`, `gcn`, `gat`, `graphsage`, `gine`
- edge-attribute flag variations (where meaningful)
- attention: none / gated / gated_projected
- SSL: off / on

For each scenario it checks output shapes and finite values.

In [13]:
def _to_plain_cfg(cfg_node):
    plain = OmegaConf.to_container(cfg_node, resolve=True)
    if isinstance(plain, dict):
        plain.pop("_target_", None)
    return plain


def run_model_scenario(unit_graph, scenario):
    cfg_local = copy.deepcopy(cfg)

    cfg_local.model.encoder.conv_type = scenario["conv_type"]
    cfg_local.model.encoder.use_edge_attr = scenario["use_edge_attr"]
    cfg_local.model.flags.use_attention = scenario["use_attention"]
    cfg_local.model.flags.use_ssl = scenario["use_ssl"]

    encoder_cfg = hydra.utils.instantiate(cfg_local.model.encoder)
    encoder_cfg.input_dim = int(unit_graph.x.size(1))

    edge_attr_local = unit_graph.edge_attr if hasattr(unit_graph, "edge_attr") else None
    if encoder_cfg.conv_type == "gine":
        if edge_attr_local is None:
            raise ValueError("Scenario requested GINE but unit has no edge_attr")
        encoder_cfg.edge_attr_dim = int(edge_attr_local.size(1))

    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        encoder_local = build_encoder(encoder_cfg)

    node_emb_local, graph_emb_local = encoder_local(
        unit_graph.x,
        unit_graph.edge_index,
        edge_attr=edge_attr_local,
        batch=getattr(unit_graph, "batch", None),
        return_graph_embedding=True,
    )

    assert torch.isfinite(node_emb_local).all()
    assert torch.isfinite(graph_emb_local).all()

    graph_head_cfg_local = _to_plain_cfg(cfg_local.model.graph_head)
    node_head_cfg_local = _to_plain_cfg(cfg_local.model.node_head)
    graph_logits_local = build_graph_head(graph_head_cfg_local)(graph_emb_local)
    node_logits_local = build_node_head(node_head_cfg_local)(node_emb_local)
    assert torch.isfinite(graph_logits_local).all()
    assert torch.isfinite(node_logits_local).all()

    attn_shape = None
    if scenario["use_attention"]:
        attention_cfg_local = _to_plain_cfg(cfg_local.model.attention)
        if scenario["attention_type"] == "gated":
            attention_cfg_local.pop("projection_dim", None)
        attn = build_attention(
            use_attention=True,
            attention_type=scenario["attention_type"],
            **attention_cfg_local,
        )
        attn_logits_local, _ = attn(node_emb_local)
        attn_shape = tuple(attn_logits_local.shape)
        assert torch.isfinite(attn_logits_local).all()

    ssl_shape = None
    if scenario["use_ssl"]:
        ssl_cfg_local = _to_plain_cfg(cfg_local.model.ssl)
        predictor_cfg_local = ssl_cfg_local.get("predictor", {})
        if isinstance(predictor_cfg_local, dict):
            predictor_cfg_local.pop("_target_", None)
        ssl_model_local = build_ssl(
            use_ssl=True,
            encoder=encoder_local,
            ssl_config=ssl_cfg_local,
        )
        q_local, y_local = ssl_model_local(
            (unit_graph.x, unit_graph.edge_index, edge_attr_local),
            (unit_graph.x, unit_graph.edge_index, edge_attr_local),
        )
        ssl_shape = (tuple(q_local.shape), tuple(y_local.shape))
        assert torch.isfinite(q_local).all()
        assert torch.isfinite(y_local).all()

    return {
        "scenario": scenario,
        "node_emb": tuple(node_emb_local.shape),
        "graph_emb": tuple(graph_emb_local.shape),
        "graph_logits": tuple(graph_logits_local.shape),
        "node_logits": tuple(node_logits_local.shape),
        "attention_logits": attn_shape,
        "ssl_shapes": ssl_shape,
        "warnings": [str(w.message) for w in caught],
    }


scenarios = [
    {
        "conv_type": "gin",
        "use_edge_attr": False,
        "use_attention": False,
        "attention_type": "gated",
        "use_ssl": False,
    },
    {
        "conv_type": "gin",
        "use_edge_attr": False,
        "use_attention": True,
        "attention_type": "gated",
        "use_ssl": True,
    },
    {
        "conv_type": "gcn",
        "use_edge_attr": False,
        "use_attention": True,
        "attention_type": "gated_projected",
        "use_ssl": False,
    },
    # {
    #     "conv_type": "gcn",
    #     "use_edge_attr": True,
    #     "use_attention": False,
    #     "attention_type": "gated",
    #     "use_ssl": False,
    # },
    {
        "conv_type": "gat",
        "use_edge_attr": False,
        "use_attention": True,
        "attention_type": "gated",
        "use_ssl": False,
    },
    {
        "conv_type": "gat",
        "use_edge_attr": True,
        "use_attention": False,
        "attention_type": "gated",
        "use_ssl": False,
    },
    {
        "conv_type": "graphsage",
        "use_edge_attr": False,
        "use_attention": True,
        "attention_type": "gated_projected",
        "use_ssl": False,
    },
    {
        "conv_type": "graphsage",
        "use_edge_attr": True,
        "use_attention": False,
        "attention_type": "gated",
        "use_ssl": False,
    },
    {
        "conv_type": "gine",
        "use_edge_attr": True,
        "use_attention": True,
        "attention_type": "gated_projected",
        "use_ssl": False,
    },
    {
        "conv_type": "gine",
        "use_edge_attr": True,
        "use_attention": False,
        "attention_type": "gated",
        "use_ssl": True,
    },
]

results = []
for idx, scenario in enumerate(scenarios, start=1):
    try:
        result = run_model_scenario(unit, scenario)
        results.append(result)
        print(f"[{idx}] OK: {scenario}")
    except Exception as exc:
        print(f"[{idx}] FAILED: {scenario}")
        raise

print(f"Completed {len(results)} scenarios.")

[1] OK: {'conv_type': 'gin', 'use_edge_attr': False, 'use_attention': False, 'attention_type': 'gated', 'use_ssl': False}
[2] OK: {'conv_type': 'gin', 'use_edge_attr': False, 'use_attention': True, 'attention_type': 'gated', 'use_ssl': True}
[3] OK: {'conv_type': 'gcn', 'use_edge_attr': False, 'use_attention': True, 'attention_type': 'gated_projected', 'use_ssl': False}
[4] OK: {'conv_type': 'gat', 'use_edge_attr': False, 'use_attention': True, 'attention_type': 'gated', 'use_ssl': False}
[5] OK: {'conv_type': 'gat', 'use_edge_attr': True, 'use_attention': False, 'attention_type': 'gated', 'use_ssl': False}
[6] OK: {'conv_type': 'graphsage', 'use_edge_attr': False, 'use_attention': True, 'attention_type': 'gated_projected', 'use_ssl': False}
[7] OK: {'conv_type': 'graphsage', 'use_edge_attr': True, 'use_attention': False, 'attention_type': 'gated', 'use_ssl': False}
[8] OK: {'conv_type': 'gine', 'use_edge_attr': True, 'use_attention': True, 'attention_type': 'gated_projected', 'use_ssl

In [14]:
for i, item in enumerate(results, start=1):
    print("-" * 100)
    print(f"Scenario {i}: {item['scenario']}")
    print(f"node_emb={item['node_emb']}, graph_emb={item['graph_emb']}")
    print(f"node_logits={item['node_logits']}, graph_logits={item['graph_logits']}")
    print(
        f"attention_logits={item['attention_logits']}, ssl_shapes={item['ssl_shapes']}"
    )
    if item["warnings"]:
        print("warnings:")
        for w in item["warnings"]:
            print(f"  - {w}")

----------------------------------------------------------------------------------------------------
Scenario 1: {'conv_type': 'gin', 'use_edge_attr': False, 'use_attention': False, 'attention_type': 'gated', 'use_ssl': False}
node_emb=(64, 128), graph_emb=(1, 128)
node_logits=(64, 1), graph_logits=(1, 1)
attention_logits=None, ssl_shapes=None
----------------------------------------------------------------------------------------------------
Scenario 2: {'conv_type': 'gin', 'use_edge_attr': False, 'use_attention': True, 'attention_type': 'gated', 'use_ssl': True}
node_emb=(64, 128), graph_emb=(1, 128)
node_logits=(64, 1), graph_logits=(1, 1)
attention_logits=(64, 1), ssl_shapes=((64, 128), (64, 128))
----------------------------------------------------------------------------------------------------
Scenario 3: {'conv_type': 'gcn', 'use_edge_attr': False, 'use_attention': True, 'attention_type': 'gated_projected', 'use_ssl': False}
node_emb=(64, 128), graph_emb=(1, 128)
node_logits=(6